In [8]:
from pathlib import Path
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name != "DataCompetition":
    PROJECT_ROOT = Path.home() / "Documents" / "DataCompetition"

DATA_DIR = PROJECT_ROOT / "data"
train = pd.read_csv(DATA_DIR / "train.csv")

TARGET = "Will_Buy_EV"
ID_COL = "id"

NUM_COLS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

CAT_COLS = [
    "Gender",
    "City_Type",
    "Current_Car_Type",
    "Home_Charging_Possible",
    "Subsidy_Available",
    "Range_Anxiety_Level",
]

X_raw = train.drop(columns=[TARGET, ID_COL])
y = train[TARGET].map({"No": 0, "Yes": 1}).astype(np.int8)

BASE_X = pd.get_dummies(
    X_raw,
    columns=CAT_COLS,
    dtype=np.int8,
)

print(f"Train shape: {train.shape}")
print(f"Base feature count: {BASE_X.shape[1]}")
print(f"Target mean: {y.mean():.6f}")


Train shape: (668665, 15)
Base feature count: 24
Target mean: 0.174645


In [9]:
RANDOM_STATE = 42
N_SPLITS = 3
SMOOTHING = 20.0

XGB_PARAMS = dict(
    n_estimators=1000,
    max_depth=5,
    learning_rate=0.035,
    min_child_weight=2,
    subsample=0.90,
    colsample_bytree=0.85,
    gamma=0,
    reg_alpha=0,
    reg_lambda=1,
    objective="binary:logistic",
    eval_metric="auc",
    tree_method="hist",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)


In [10]:
IDENTITY_COLS = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

IDENTITY_PAIRS = [
    ("Age", "Annual_Income_USD"),
    ("Age", "Daily_Commute_km"),
    ("Age", "Current_Car_Type"),
    ("Annual_Income_USD", "Current_Car_Type"),
    ("Annual_Income_USD", "City_Type"),
    ("Daily_Commute_km", "Current_Car_Type"),
    ("Charging_Stations_Near_Home", "Charging_Stations_Near_Work"),
    ("Environmental_Concern_Level", "Range_Anxiety_Level"),
]


In [11]:
def add_identity_features_oof(base_X, raw_X, y, columns):
    out = base_X.copy()

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    global_mean = y.mean()

    for col in columns:
        values = raw_X[col]
        te = np.zeros(len(raw_X))
        freq = np.zeros(len(raw_X))

        for tr_idx, va_idx in skf.split(raw_X, y):
            train_values = values.iloc[tr_idx]
            valid_values = values.iloc[va_idx]
            train_y = y.iloc[tr_idx]

            stats = pd.DataFrame({
                "key": train_values.to_numpy(),
                "target": train_y.to_numpy(),
            })

            grouped = (
                stats.groupby("key", dropna=False)["target"]
                .agg(["sum", "count"])
            )

            sums = (
                valid_values.map(grouped["sum"])
                .fillna(0)
                .to_numpy()
            )

            counts = (
                valid_values.map(grouped["count"])
                .fillna(0)
                .to_numpy()
            )

            te[va_idx] = (
                sums + SMOOTHING * global_mean
            ) / (
                counts + SMOOTHING
            )

            freq[va_idx] = counts / len(tr_idx)

        out[f"{col}__te"] = te
        out[f"{col}__freq"] = freq

    return out


In [12]:
def add_pair_features_oof(base_X, raw_X, y, pairs):
    out = base_X.copy()

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    global_mean = y.mean()

    for cols in pairs:
        name = "__".join(cols)

        keys = (
            raw_X[list(cols)]
            .astype(str)
            .agg("||".join, axis=1)
        )

        te = np.zeros(len(raw_X))
        freq = np.zeros(len(raw_X))

        for tr_idx, va_idx in skf.split(raw_X, y):
            train_keys = keys.iloc[tr_idx]
            valid_keys = keys.iloc[va_idx]
            train_y = y.iloc[tr_idx]

            stats = pd.DataFrame({
                "key": train_keys.to_numpy(),
                "target": train_y.to_numpy(),
            })

            grouped = (
                stats.groupby("key", dropna=False)["target"]
                .agg(["sum", "count"])
            )

            sums = (
                valid_keys.map(grouped["sum"])
                .fillna(0)
                .to_numpy()
            )

            counts = (
                valid_keys.map(grouped["count"])
                .fillna(0)
                .to_numpy()
            )

            te[va_idx] = (
                sums + SMOOTHING * global_mean
            ) / (
                counts + SMOOTHING
            )

            freq[va_idx] = counts / len(tr_idx)

        out[f"{name}__te"] = te
        out[f"{name}__freq"] = freq

    return out


In [13]:
X_identity = add_identity_features_oof(
    BASE_X,
    X_raw,
    y,
    IDENTITY_COLS,
)

X_identity = add_pair_features_oof(
    X_identity,
    X_raw,
    y,
    IDENTITY_PAIRS,
)

print(f"Base features: {BASE_X.shape[1]}")
print(f"Identity features: {X_identity.shape[1] - BASE_X.shape[1]}")
print(f"Final feature count: {X_identity.shape[1]}")


Base features: 24
Identity features: 30
Final feature count: 54


In [14]:
skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE,
)

oof = np.zeros(len(X_identity))

for fold, (tr_idx, va_idx) in enumerate(
    skf.split(X_identity, y),
    start=1,
):
    print(f"Training fold {fold}/{N_SPLITS}...")

    model = XGBClassifier(**XGB_PARAMS)

    model.fit(
        X_identity.iloc[tr_idx],
        y.iloc[tr_idx],
    )

    oof[va_idx] = model.predict_proba(
        X_identity.iloc[va_idx]
    )[:, 1]

    fold_auc = roc_auc_score(
        y.iloc[va_idx],
        oof[va_idx],
    )

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")

overall_auc = roc_auc_score(y, oof)

print()
print(f"Experiment 25 ROC-AUC: {overall_auc:.6f}")
print("Previous best: 0.941731")
print(f"Difference: {overall_auc - 0.941731:+.6f}")


Training fold 1/3...
Fold 1 ROC-AUC: 0.943380
Training fold 2/3...
Fold 2 ROC-AUC: 0.943206
Training fold 3/3...
Fold 3 ROC-AUC: 0.944199

Experiment 25 ROC-AUC: 0.943065
Previous best: 0.941731
Difference: +0.001334


In [15]:
results = pd.DataFrame({
    "experiment": ["Experiment 25"],
    "roc_auc": [overall_auc],
    "previous_best": [0.941731],
    "difference": [overall_auc - 0.941731],
})

results


,experiment,roc_auc,previous_best,difference
0,Experiment 25,0.943065,0.941731,0.001334
